# Landsat 8 Change-Detection Coding Challenge

**Area:** Noida scene, Path 146 / Row 040  
**Dates:** 13 December 2015 and 30 July 2024

This notebook uses the real SR_B2, SR_B3, SR_B4, SR_B5 and QA_PIXEL files. It demonstrates preprocessing, Maximum Likelihood Estimation, linear regression and possible-change classification. MLE training samples are provisional spectral-rule samples, not ground truth. Therefore, true accuracy is not reported.


## 1. Install packages

Run this once in a new Colab session. Restarting the runtime is normally unnecessary.


In [ ]:
%pip -q install rasterio numpy matplotlib scikit-learn

## 2. Connect Google Drive and set the data folder

Upload all ten `.TIF` files into one Google Drive folder. Change only `COLAB_DATA_DIR` if your folder has a different name.


In [ ]:
from pathlib import Path
import os

# Local: launch Jupyter from the cloned repository.
# Colab: upload/extract landsat-inputs-v1.zip into /content first,
# or set LANDSAT_DATA_DIR to your extracted Drive folder.
DATA_DIR = Path(os.environ.get("LANDSAT_DATA_DIR", "colab_test_data")).resolve()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Extract landsat-inputs-v1.zip first, or set LANDSAT_DATA_DIR: {DATA_DIR}")
print("Data folder:", DATA_DIR)


## 3. Imports and file discovery

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from sklearn.linear_model import LinearRegression

DATES = {"2015": "20151213", "2024": "20240730"}
BANDS = ("SR_B2", "SR_B3", "SR_B4", "SR_B5")
BAD_QA_BITS = sum(1 << bit for bit in range(6))
SCALE = 0.0000275
OFFSET = -0.2
GRID_FACTOR = 5  # 30 m source -> 150 m demonstration grid
RNG = np.random.default_rng(42)

def locate(date_token, band):
    matches = sorted(DATA_DIR.glob(f"LC08_L2SP_*_{date_token}_*_{band}.TIF"))
    if len(matches) != 1:
        raise ValueError(f"Expected one {date_token} {band} file, found {len(matches)}")
    return matches[0]

FILES = {year: {band: locate(token, band) for band in (*BANDS, "QA_PIXEL")}
         for year, token in DATES.items()}

for year, items in FILES.items():
    print(f"{year}:")
    for band, path in items.items():
        print(f"  {band}: {path.name}")


## 4. Scaling, QA masking and geographic alignment

QA bits 0–5 remove fill, dilated cloud, cirrus, cloud, cloud shadow and snow. The 2024 files are reprojected onto the 2015 B4 reference grid. Nearest-neighbour resampling preserves the discrete QA flags.


In [ ]:
def make_reference_grid():
    with rasterio.open(FILES["2015"]["SR_B4"]) as src:
        height = (src.height + GRID_FACTOR - 1) // GRID_FACTOR
        width = (src.width + GRID_FACTOR - 1) // GRID_FACTOR
        transform = src.transform * src.transform.scale(src.width / width, src.height / height)
        return {"crs": src.crs, "transform": transform, "height": height,
                "width": width, "source_shape": (src.height, src.width),
                "resolution": src.res, "bounds": src.bounds}

TARGET = make_reference_grid()

def read_to_reference(path):
    output = np.zeros((TARGET["height"], TARGET["width"]), dtype=np.float32)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), output,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                  dst_transform=TARGET["transform"], dst_crs=TARGET["crs"], dst_nodata=0,
                  resampling=Resampling.nearest)
    return output

def load_year(year):
    qa = read_to_reference(FILES[year]["QA_PIXEL"]).astype(np.uint16)
    qa_valid = (qa & BAD_QA_BITS) == 0
    layers = []
    for band in BANDS:
        dn = read_to_reference(FILES[year][band])
        valid = qa_valid & (dn > 0)
        reflectance = dn * np.float32(SCALE) + np.float32(OFFSET)
        reflectance[~valid] = np.nan
        layers.append(reflectance)
    stack = np.stack(layers)
    return stack, np.isfinite(stack).all(axis=0)

image_2015, valid_2015 = load_year("2015")
image_2024, valid_2024 = load_year("2024")
common_valid = valid_2015 & valid_2024

print("CRS:", TARGET["crs"])
print("Native shape:", TARGET["source_shape"])
print("Native pixel size:", TARGET["resolution"])
print("Analysis shape:", common_valid.shape)
print("Common valid analysis pixels:", f"{common_valid.sum():,}")


## 5. Natural-colour visualization

In [ ]:
def rgb(stack, valid):
    image = np.moveaxis(stack[[2, 1, 0]], 0, -1)  # B4, B3, B2
    image = np.clip(np.nan_to_num(image) / 0.30, 0, 1)
    return np.dstack([image, valid.astype(np.float32)])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
axes[0].imshow(rgb(image_2015, common_valid))
axes[0].set_title("2015 natural colour")
axes[1].imshow(rgb(image_2024, common_valid))
axes[1].set_title("2024 natural colour")
for ax in axes:
    ax.axis("off")
plt.show()


## 6. Maximum Likelihood Estimation

The four reflectance bands are the features. Spectral rules provide provisional seed samples so the required MLE concept can be executed without invented ground-truth labels.


In [ ]:
def create_seed_masks(stack, valid):
    blue, green, red, nir = stack
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)
    visible = stack[:3]
    finite = np.isfinite(visible)
    brightness = np.divide(np.nansum(visible, axis=0), finite.sum(axis=0),
                           out=np.full(valid.shape, np.nan, dtype=np.float32),
                           where=finite.sum(axis=0) > 0)
    nq = np.quantile(ndvi[valid], [0.30, 0.55, 0.80])
    wq = np.quantile(ndwi[valid], 0.90)
    bq = np.quantile(brightness[valid], [0.45, 0.75])
    return {
        "Water-like": valid & (ndwi >= wq) & (ndvi <= nq[0]),
        "Vegetation-like": valid & (ndvi >= nq[2]),
        "Bright low-vegetation": valid & (brightness >= bq[1]) & (ndvi <= nq[1]),
        "Bare/other-like": valid & (brightness >= bq[0]) & (ndvi > nq[0]) & (ndvi < nq[2]),
    }

def train_mle(stack, masks):
    pixels = np.moveaxis(stack, 0, -1)
    models = []
    for class_id, (name, mask) in enumerate(masks.items()):
        samples = pixels[mask]
        print(f"{name}: {len(samples):,} provisional samples")
        if len(samples) > 5000:
            samples = samples[RNG.choice(len(samples), 5000, replace=False)]
        covariance = np.cov(samples, rowvar=False) + np.eye(4) * 1e-6
        models.append((class_id, name, samples.mean(axis=0),
                       np.linalg.pinv(covariance), np.linalg.slogdet(covariance)[1]))
    return models

def mle_predict(stack, valid, models):
    pixels = np.moveaxis(stack, 0, -1)
    positions = np.flatnonzero(valid)
    x = pixels.reshape(-1, 4)[positions]
    best = np.full(len(x), -np.inf)
    labels = np.full(len(x), -1, dtype=np.int8)
    for class_id, name, mean, inverse_covariance, log_determinant in models:
        difference = x - mean
        score = -0.5 * (log_determinant + np.einsum(
            "ij,jk,ik->i", difference, inverse_covariance, difference))
        select = score > best
        labels[select], best[select] = class_id, score[select]
    result = np.full(valid.shape, -1, dtype=np.int8)
    result.flat[positions] = labels
    return result

seed_masks = create_seed_masks(image_2015, valid_2015)
mle_models = train_mle(image_2015, seed_masks)
mle_2015 = mle_predict(image_2015, valid_2015, mle_models)
mle_2024 = mle_predict(image_2024, valid_2024, mle_models)
print("MLE classification completed.")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np

# Colours for the four MLE classes
class_names = [
    "Water-like",
    "Vegetation-like",
    "Bright low-vegetation",
    "Bare/other-like"
]

class_colors = [
    "#2b83ba",  # blue
    "#1a9850",  # green
    "#d8b365",  # light brown
    "#a6611a"   # dark brown
]

mle_cmap = ListedColormap(class_colors)
mle_cmap.set_bad("#d9d9d9")

# Hide invalid pixels, which have the value -1
display_2015 = np.ma.masked_less(mle_2015, 0)
display_2024 = np.ma.masked_less(mle_2024, 0)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(
    display_2015,
    cmap=mle_cmap,
    vmin=0,
    vmax=3,
    interpolation="nearest"
)
axes[0].set_title("2015 MLE Classification")
axes[0].axis("off")

axes[1].imshow(
    display_2024,
    cmap=mle_cmap,
    vmin=0,
    vmax=3,
    interpolation="nearest"
)
axes[1].set_title("2024 MLE Classification")
axes[1].axis("off")

legend_items = [
    Patch(facecolor=color, label=name)
    for name, color in zip(class_names, class_colors)
]

fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=4,
    bbox_to_anchor=(0.5, -0.02)
)

fig.suptitle("Maximum Likelihood Classification", fontsize=16)
plt.tight_layout(rect=[0, 0.07, 1, 0.94])

plt.savefig(
    "mle_classification_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Graph saved as: mle_classification_comparison.png")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np

class_names = [
    "Water-like",
    "Vegetation-like",
    "Bright low-vegetation",
    "Bare/other-like"
]

class_colors = [
    "#2b83ba",
    "#1a9850",
    "#d8b365",
    "#a6611a"
]

mle_cmap = ListedColormap(class_colors)
mle_cmap.set_bad("#d9d9d9")

display_2015 = np.ma.masked_less(mle_2015, 0)
display_2024 = np.ma.masked_less(mle_2024, 0)

# Count classified pixels
counts_2015 = np.bincount(
    mle_2015[mle_2015 >= 0],
    minlength=4
)

counts_2024 = np.bincount(
    mle_2024[mle_2024 >= 0],
    minlength=4
)

total_2015 = counts_2015.sum()
total_2024 = counts_2024.sum()

percent_2015 = counts_2015 / total_2015 * 100
percent_2024 = counts_2024 / total_2024 * 100

# Create figure
fig = plt.figure(figsize=(17, 12))
grid = fig.add_gridspec(2, 2, height_ratios=[2, 1])

ax1 = fig.add_subplot(grid[0, 0])
ax2 = fig.add_subplot(grid[0, 1])
ax3 = fig.add_subplot(grid[1, :])

# MLE maps
ax1.imshow(
    display_2015,
    cmap=mle_cmap,
    vmin=0,
    vmax=3,
    interpolation="nearest"
)
ax1.set_title(
    f"2015 MLE Classification\n"
    f"Classified pixels: {total_2015:,}"
)
ax1.axis("off")

ax2.imshow(
    display_2024,
    cmap=mle_cmap,
    vmin=0,
    vmax=3,
    interpolation="nearest"
)
ax2.set_title(
    f"2024 MLE Classification\n"
    f"Classified pixels: {total_2024:,}"
)
ax2.axis("off")

# Percentage comparison
x = np.arange(len(class_names))
bar_width = 0.36

bars_2015 = ax3.bar(
    x - bar_width / 2,
    percent_2015,
    bar_width,
    label="2015",
    color="#4878A8"
)

bars_2024 = ax3.bar(
    x + bar_width / 2,
    percent_2024,
    bar_width,
    label="2024",
    color="#E07B39"
)

# Add percentage and pixel count above each bar
for bars, percentages, counts in [
    (bars_2015, percent_2015, counts_2015),
    (bars_2024, percent_2024, counts_2024)
]:
    for bar, percentage, count in zip(bars, percentages, counts):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{percentage:.1f}%\n({count:,})",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax3.set_xticks(x)
ax3.set_xticklabels(class_names)
ax3.set_ylabel("Percentage of classified pixels")
ax3.set_title("MLE Class Distribution: 2015 and 2024")
ax3.legend()
ax3.grid(axis="y", linestyle="--", alpha=0.3)
ax3.set_ylim(
    0,
    max(percent_2015.max(), percent_2024.max()) + 10
)

legend_items = [
    Patch(facecolor=color, label=name)
    for name, color in zip(class_names, class_colors)
]

fig.legend(
    handles=legend_items,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=4
)

fig.suptitle(
    "Maximum Likelihood Classification Comparison",
    fontsize=17
)

plt.tight_layout(rect=[0, 0, 1, 0.91])

plt.savefig(
    "mle_comparison_with_numbers.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("2015 counts:", counts_2015)
print("2024 counts:", counts_2024)
print("Saved: mle_comparison_with_numbers.png")

## 7. Linear regression and possible-change classification

Regression is fitted separately for B2–B5 using 100,000 corresponding valid pixels. A robust residual threshold classifies possible spectral change. This is not verified land-cover change.


In [ ]:
old_pixels = np.moveaxis(image_2015, 0, -1)[common_valid]
new_pixels = np.moveaxis(image_2024, 0, -1)[common_valid]
sample_count = min(100_000, len(old_pixels))
sample_indices = RNG.choice(len(old_pixels), sample_count, replace=False)
residuals = np.empty_like(new_pixels, dtype=np.float32)
regression_results = {}

for index, band in enumerate(BANDS):
    model = LinearRegression().fit(old_pixels[sample_indices, index, None],
                                   new_pixels[sample_indices, index])
    prediction = model.predict(old_pixels[:, index, None])
    residuals[:, index] = new_pixels[:, index] - prediction
    regression_results[band] = {
        "slope": model.coef_[0], "intercept": model.intercept_,
        "r2": model.score(old_pixels[sample_indices, index, None],
                          new_pixels[sample_indices, index]),
        "rmse": np.sqrt(np.mean(residuals[:, index] ** 2)),
    }

print(f"{'Band':<8} {'Slope':>9} {'Intercept':>11} {'R2':>9} {'RMSE':>9}")
for band, result in regression_results.items():
    print(f"{band:<8} {result['slope']:>9.4f} {result['intercept']:>11.4f} "
          f"{result['r2']:>9.4f} {result['rmse']:>9.4f}")

residual_scale = np.median(np.abs(residuals), axis=0) + 1e-6
scores = np.sqrt(np.mean((residuals / residual_scale) ** 2, axis=1))
score_median = np.median(scores)
score_mad = np.median(np.abs(scores - score_median))
threshold = score_median + 3 * 1.4826 * score_mad
possible_change = scores > threshold

score_map = np.full(common_valid.shape, np.nan, dtype=np.float32)
change_map = np.full(common_valid.shape, -1, dtype=np.int8)
score_map[common_valid] = scores
change_map[common_valid] = possible_change.astype(np.int8)

print("Residual threshold:", round(float(threshold), 4))
print("Possible-change pixels:", f"{possible_change.sum():,}")
print("Possible-change percentage:", f"{100 * possible_change.mean():.2f}%")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

# Use a smaller sample for clear scatter plots
plot_count = min(3000, sample_count)
plot_indices = sample_indices[:plot_count]

fig = plt.figure(figsize=(20, 12))
grid = fig.add_gridspec(2, 4, height_ratios=[1, 1.15])

# -----------------------------
# Band-wise regression graphs
# -----------------------------
for index, band in enumerate(BANDS):
    ax = fig.add_subplot(grid[0, index])

    x_values = old_pixels[plot_indices, index]
    y_values = new_pixels[plot_indices, index]

    result = regression_results[band]

    line_x = np.linspace(
        x_values.min(),
        x_values.max(),
        100
    )

    line_y = (
        result["intercept"]
        + result["slope"] * line_x
    )

    ax.scatter(
        x_values,
        y_values,
        s=7,
        alpha=0.25,
        color="#3366A0",
        label="Sample pixels"
    )

    ax.plot(
        line_x,
        line_y,
        color="#D62728",
        linewidth=2,
        label="Regression line"
    )

    ax.set_title(
        f"{band}\n"
        f"Slope = {result['slope']:.3f}\n"
        f"R² = {result['r2']:.3f}, "
        f"RMSE = {result['rmse']:.3f}"
    )

    ax.set_xlabel("2015 reflectance")
    ax.set_ylabel("2024 reflectance")
    ax.grid(alpha=0.2)

    if index == 0:
        ax.legend()

# -----------------------------
# Residual score map
# -----------------------------
ax_residual = fig.add_subplot(grid[1, 0:2])

residual_display_limit = np.nanpercentile(
    score_map,
    98
)

residual_image = ax_residual.imshow(
    score_map,
    cmap="magma",
    vmin=0,
    vmax=residual_display_limit
)

ax_residual.set_title(
    "Regression Residual Score\n"
    f"Classification threshold = {threshold:.4f}"
)

ax_residual.axis("off")

colorbar = fig.colorbar(
    residual_image,
    ax=ax_residual,
    fraction=0.035,
    pad=0.02
)

colorbar.set_label("Standardized residual score")

# -----------------------------
# Possible-change map
# -----------------------------
ax_change = fig.add_subplot(grid[1, 2])

change_cmap = ListedColormap([
    "#202020",  # unchanged
    "#E31A1C"   # possible change
])

change_cmap.set_bad("#D9D9D9")

masked_change_map = np.ma.masked_less(
    change_map,
    0
)

ax_change.imshow(
    masked_change_map,
    cmap=change_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest"
)

changed_count = int(possible_change.sum())
unchanged_count = int(len(possible_change) - changed_count)
total_count = len(possible_change)

changed_percentage = changed_count / total_count * 100
unchanged_percentage = unchanged_count / total_count * 100

ax_change.set_title(
    "Possible-Change Classification\n"
    f"Changed: {changed_count:,} "
    f"({changed_percentage:.2f}%)"
)

ax_change.axis("off")

# -----------------------------
# Changed/unchanged bar graph
# -----------------------------
ax_bar = fig.add_subplot(grid[1, 3])

categories = ["Unchanged", "Possible\nchange"]
counts = [unchanged_count, changed_count]
percentages = [
    unchanged_percentage,
    changed_percentage
]

bars = ax_bar.bar(
    categories,
    percentages,
    color=["#333333", "#E31A1C"],
    width=0.6
)

for bar, count, percentage in zip(
    bars,
    counts,
    percentages
):
    ax_bar.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{percentage:.2f}%\n({count:,} pixels)",
        ha="center",
        va="bottom",
        fontsize=10
    )

ax_bar.set_ylabel("Percentage of common valid pixels")
ax_bar.set_title(
    "Change Classification Summary\n"
    f"Total valid pixels: {total_count:,}"
)

ax_bar.set_ylim(
    0,
    max(percentages) + 12
)

ax_bar.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

fig.suptitle(
    "Linear Regression and Possible-Change Classification",
    fontsize=18
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

plt.savefig(
    "regression_change_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Regression sample size:", f"{sample_count:,}")
print("Residual threshold:", f"{threshold:.4f}")
print("Unchanged pixels:", f"{unchanged_count:,}")
print("Possible-change pixels:", f"{changed_count:,}")
print("Possible-change percentage:", f"{changed_percentage:.2f}%")
print("Saved: regression_change_comparison.png")

## 8. Final Matplotlib results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np

# --------------------------------------------------
# Colours and class names
# --------------------------------------------------
class_names = [model[1] for model in mle_models]

class_colors = [
    "#2b83ba",
    "#1a9850",
    "#d8b365",
    "#a6611a"
]

class_cmap = ListedColormap(class_colors)
class_cmap.set_bad("#d9d9d9")

change_cmap = ListedColormap([
    "#202020",  # unchanged
    "#e31a1c"   # possible change
])
change_cmap.set_bad("#d9d9d9")

# --------------------------------------------------
# Crop maps to the common valid-data area
# --------------------------------------------------
rows, columns = np.where(common_valid)

margin = 20

row_start = max(0, rows.min() - margin)
row_end = min(common_valid.shape[0], rows.max() + margin)

column_start = max(0, columns.min() - margin)
column_end = min(common_valid.shape[1], columns.max() + margin)

crop_rows = slice(row_start, row_end)
crop_columns = slice(column_start, column_end)

def crop(array):
    return array[crop_rows, crop_columns]

# Use the same valid footprint for both MLE maps
mle_2015_common = np.where(
    common_valid,
    mle_2015,
    -1
)

mle_2024_common = np.where(
    common_valid,
    mle_2024,
    -1
)

# --------------------------------------------------
# Comparable class counts
# --------------------------------------------------
counts_2015 = np.bincount(
    mle_2015[common_valid],
    minlength=4
)

counts_2024 = np.bincount(
    mle_2024[common_valid],
    minlength=4
)

total_valid = int(common_valid.sum())

percent_2015 = counts_2015 / total_valid * 100
percent_2024 = counts_2024 / total_valid * 100

changed_count = int(np.sum(change_map == 1))
unchanged_count = int(np.sum(change_map == 0))

changed_percentage = changed_count / total_valid * 100
unchanged_percentage = unchanged_count / total_valid * 100

# ==================================================
# FIGURE 1: CLEAN MAP COMPARISON
# ==================================================
fig, axes = plt.subplots(
    2,
    3,
    figsize=(22, 14)
)

fig.patch.set_facecolor("white")

# Natural-colour images
axes[0, 0].imshow(
    crop(rgb(image_2015, common_valid))
)
axes[0, 0].set_title(
    "2015 Natural-Colour Image",
    fontsize=16,
    fontweight="bold"
)

axes[0, 1].imshow(
    crop(rgb(image_2024, common_valid))
)
axes[0, 1].set_title(
    "2024 Natural-Colour Image",
    fontsize=16,
    fontweight="bold"
)

# MLE maps
axes[0, 2].imshow(
    np.ma.masked_less(crop(mle_2015_common), 0),
    cmap=class_cmap,
    vmin=0,
    vmax=3,
    interpolation="nearest"
)
axes[0, 2].set_title(
    "2015 MLE Classification",
    fontsize=16,
    fontweight="bold"
)

axes[1, 0].imshow(
    np.ma.masked_less(crop(mle_2024_common), 0),
    cmap=class_cmap,
    vmin=0,
    vmax=3,
    interpolation="nearest"
)
axes[1, 0].set_title(
    "2024 MLE Classification",
    fontsize=16,
    fontweight="bold"
)

# Residual score
residual_limit = np.nanpercentile(
    score_map,
    98
)

residual_image = axes[1, 1].imshow(
    crop(score_map),
    cmap="magma",
    vmin=0,
    vmax=residual_limit
)

axes[1, 1].set_title(
    f"Regression Residual Score\nThreshold = {threshold:.4f}",
    fontsize=16,
    fontweight="bold"
)

residual_colorbar = fig.colorbar(
    residual_image,
    ax=axes[1, 1],
    fraction=0.046,
    pad=0.03
)

residual_colorbar.set_label(
    "Residual score",
    fontsize=12
)

# Change map
axes[1, 2].imshow(
    np.ma.masked_less(crop(change_map), 0),
    cmap=change_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest"
)

axes[1, 2].set_title(
    f"Possible-Change Map\n"
    f"{changed_count:,} pixels ({changed_percentage:.2f}%)",
    fontsize=16,
    fontweight="bold"
)

for axis in axes.flat:
    axis.axis("off")

# Clear legends
class_legend = [
    Patch(facecolor=color, label=name)
    for name, color in zip(class_names, class_colors)
]

change_legend = [
    Patch(facecolor="#202020", label="Unchanged"),
    Patch(facecolor="#e31a1c", label="Possible change"),
    Patch(facecolor="#d9d9d9", label="Invalid/masked")
]

legend_one = fig.legend(
    handles=class_legend,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.955),
    ncol=4,
    fontsize=12,
    title="MLE classes",
    title_fontsize=13
)

fig.add_artist(legend_one)

fig.legend(
    handles=change_legend,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.015),
    ncol=3,
    fontsize=12
)

fig.suptitle(
    "Landsat Change-Detection Results",
    fontsize=22,
    fontweight="bold",
    y=0.995
)

plt.tight_layout(
    rect=[0.02, 0.06, 0.98, 0.91],
    h_pad=2.5,
    w_pad=2
)

plt.savefig(
    "landsat_result_maps_clear.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# ==================================================
# FIGURE 2: CLEAR NUMERICAL SUMMARY
# ==================================================
fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 7)
)

# MLE percentage comparison
x = np.arange(len(class_names))
width = 0.36

bars_2015 = axes[0].bar(
    x - width / 2,
    percent_2015,
    width,
    label="2015",
    color="#3977A8"
)

bars_2024 = axes[0].bar(
    x + width / 2,
    percent_2024,
    width,
    label="2024",
    color="#E17C3A"
)

for bars, counts, percentages in [
    (bars_2015, counts_2015, percent_2015),
    (bars_2024, counts_2024, percent_2024)
]:
    for bar, count, percentage in zip(
        bars,
        counts,
        percentages
    ):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{percentage:.1f}%\n{count:,}",
            ha="center",
            va="bottom",
            fontsize=10
        )

axes[0].set_xticks(x)
axes[0].set_xticklabels(
    class_names,
    rotation=10
)
axes[0].set_ylabel("Percentage of common valid pixels")
axes[0].set_title(
    "MLE Class Comparison",
    fontsize=16,
    fontweight="bold"
)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)
axes[0].set_ylim(
    0,
    max(percent_2015.max(), percent_2024.max()) + 12
)

# Change summary
change_categories = [
    "Unchanged",
    "Possible change"
]

change_percentages = [
    unchanged_percentage,
    changed_percentage
]

change_counts = [
    unchanged_count,
    changed_count
]

change_bars = axes[1].bar(
    change_categories,
    change_percentages,
    color=["#333333", "#e31a1c"],
    width=0.55
)

for bar, count, percentage in zip(
    change_bars,
    change_counts,
    change_percentages
):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{percentage:.2f}%\n{count:,} pixels",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

axes[1].set_ylabel("Percentage of common valid pixels")
axes[1].set_title(
    f"Change Classification\n"
    f"Total valid pixels: {total_valid:,}",
    fontsize=16,
    fontweight="bold"
)
axes[1].grid(axis="y", alpha=0.25)
axes[1].set_ylim(0, 110)

fig.suptitle(
    "Numerical Results",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "landsat_numerical_summary.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved: landsat_result_maps_clear.png")
print("Saved: landsat_numerical_summary.png")

## 9. Result statement

- MLE was executed using provisional spectral training samples.
- Linear regression was executed using corresponding valid pixels.
- Regression residuals were classified into possible change and unchanged.
- The two acquisitions are from different seasons, so seasonal effects may appear as change.
- Independent ground truth was not supplied; therefore, this notebook does not report accuracy.
